# 🌍 RADON PREDICTION SYSTEM - IRBID CITY, JORDAN

## Complete Analysis in Google Colab

This notebook performs a complete radon prediction analysis:
1. Data cleaning and preparation
2. Machine learning model development (4 models)
3. Interactive visualizations
4. Geological analysis and reporting

**⏱️ Runtime: 3-5 minutes**

---

### Instructions:
1. Click the **▶ Run** button on the left of each cell
2. Or use **Runtime → Run all** to run everything at once
3. Wait for results to appear
4. Download your files from the output

## STEP 0: Install Required Packages

In [ ]:
# Install required packages
import subprocess
import sys

print("Installing required packages...")
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy', 'folium', 'joblib']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All packages installed successfully!")

## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## STEP 2: Load and Clean Data

In [ ]:
print("="*80)
print(" "*20 + "STEP 1: DATA CLEANING AND PREPARATION")
print("="*80)

# Raw radon data
raw_data = """
Location                                        Average Radon Concetration(KBq/m^3)Year    X       Y       Rock                                Soil                   Elevation
historical city of Jerash                       2.292    2012  35.534 32.1612Limestone                           Terra Rossa                   600
Muwaqqar                                        6    2000 36.0738 31.4838Chalk Marl/Phosphatic               Yellow Desert Soil            900
Abu Nasier                                      1.3    2016 35.5254 32.0355Hard Limestone/Dolomite             Terra Rossa                  1000
Suwaileh                                        0.7    2016 35.4955 32.0058Limestone/Marl                      Deep Terra Rossa             1080
Al Rusaifah                                     2.5    2016 36.0147 32.0108Phosphorite/High uranium            Disturbed/Mining Tailin       650
Malka                                           1.53    2012 35.4707 32.4321chalky Limestone/Chert              Deep Terra Rossa              600
Al Rafid                                        3.39    2012 35.5052 32.4521chalky Limestone/Chert              Heavy Clay/Cracking soi       500
Aqraba                                          4.7    2012 35.2033 32.0739chalky Limestone/Chert              Terra Rossa                   450
Hubras                                          1.55    2011    35.5 32.3936chalky Limestone/Chert              Terra Rossa            550-600
Harta                                           1.03    2011 35.5046 32.4132chalky Limestone/Chert              Terra Rossa            500-550
Sabha                                           7.48    2016 36.2938 32.1956Basalt                              Xerolls                700-800
Um eljemal                                      5.83    2016 36.2138 32.1955Basalt                              Xerolls                700-800
Sama Alserhan                                   9.79    2016 36.1441  32.281Chalky Limestone                    Light colored dry clay 500-600
Mansourah - Mafraq                              4.09    2016 36.1014 32.2507Chalky Limestone                    Light colored dry clay 500-600
Al Manshia-Mafraq                               8.75    2016 36.0451 32.2213Limestone                           Desert soil                   700
Rehab-Mafraq                                    5.84    2016 36.0522  32.193Hard calcareous and flint           Terra Rossa/Calcium    800-900
Al Khaldiah-Mafraq                              8.70     2016 36.1853   32.11Church/Marl                         Desert soil                   700
Al Mafraq                                       4.21    2016 36.1321 32.2024Limestone                           Desert soil                   700
Balamah                                         7.75    2016 36.0511   32.14Hard calcareous and flint           Terra Rossa/Calcium    800-900
Alhamrah-Mafraq                                 7.15    2016 36.0916 32.2613Hard calcareous and flint           Terra Rossa/Calcium    800-900
Albweghdah                                      10.26    2016 36.0336 32.2809Hard calcareous and flint           Terra Rossa/Calcium    800-900
Housha                                          11.32    2016 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Muwaqqar                                        3.7    2008 36.0738 31.4838Chalk Marl/Phosphatic               Yellow Desert Soil            900
Wadi essir                                      4.1    2008 35.4639   31.56Wadi essir Limestone                Terra Rossa            600-950
Jabal Alakhdar-Jerash                           3    2012 35.5436 32.1716Limestone/Marl                      Terra Rossa                   700
Samma                                           7.4    2003 35.4127 32.3402Limestone                           Terra Rossa                   450
Al Rusaifah(abandoned phsphate mine             8.44    2002 36.0206 32.0117Phosphorite/High uranium            Disturbed/Mining Tailin       650
Amman                                           6.3    2002 35.5651 31.5008Limestone/Dolomite                  Terra Rossa            800-900
Irbid                                           4.09    2013  35.505 32.3309chalky Limestone/Chert              Terra Rossa                   600
Irbid                                           15.71    2013  35.505 32.3309chalky Limestone/Chert              Terra Rossa                   600
Irbid                                           24.19    2013  35.505 32.3309chalky Limestone/Chert              Terra Rossa                   600
Yarmouk River                                   0.48    2017 35.4303 32.4213Basalt/Limestone                    Alluvial                     -150
Al Fuhais                                       2.7    2003 35.4704 32.0014Limestone/Marl                      Terra Rossa                   900
Mazar Shamali                                   47.8    1999 35.4742 32.2756Hard Limestone                      Terra Rossa                   800
Kharja                                          1.32    2006 35.5311 32.3922Chalky Limestone                    Terra Rossa                   480
Housha                                          3.32    2008 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Housha                                          7.22    2008 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Housha                                          10.77    2008 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Housha                                          14.96    2008 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Housha                                          16.26    2008 36.0552 32.2706Hard calcareous and flint           Terra Rossa/Calcium    800-900
Soum                                            6.86    2003 35.4741 32.3513Hard Limestone                      Terra Rossa                   550
Al Rusaifah                                     7.28    2000 36.0147 32.0108Phosphorite/High uranium            Disturbed/Mining Tailin       650
Ayn jana                                        2.08    2012 35.4532 32.2024Limestone/Dolomite                  Terra Rossa            1000-1100
Umm Yanabe                                      2.36    2012 35.4556  32.222Limestone/Marl                      Terra Rossa            800-900
Ishtafena                                       3.62    2012 35.4653 32.2137Limestone/Dolomite                  Forest Soil            1000-1500
Al baag                                         1.9    2008 36.2023  32.221Limestone/Basalt                    Xerosols               650-700
Ajloun city                                     3.4    2008 35.4602 32.2029Limestone/Marl                      Terra Rossa            750-850
Al Hamra                                        15.15    2011 36.0916 32.2613Hard calcareous and flint           Terra Rossa/Calcium    800-900
Bayt Yafa                                       6.9    2017  35.472 32.3108Chalky Limestone                    Terra Rossa            550-650
Al Dajania-mafraq                               32.9    2009 36.0239  32.202Limestone/Marl/Chert                Silty-Clay                    600
Al Dajania-mafraq                               8.5    2009 36.0239  32.202Limestone/Marl/Chert                Silty-Clay                    600
Al Dajania-mafraq                               2.55    2009 36.0239  32.202Limestone/Marl/Chert                Silty-Clay                    600
Soum-irbid                                      7.7    2017 35.4742 32.3513Chalky Limestone                    Terra Rossa                   500
Just                                            0.056    1998 35.5928 32.2942Limestone                           Vertisols              580-610
Ramtha                                          0.062    1998  36.001 32.3227Limestone                           Vertisols              500-530
Torra                                           0.062    1998  35.592 32.3805Chalky Marl                         Marly Soil             450-480
Shajera                                         0.063    1998 35.5623 32.3836Chalky Marl                         Deep cultivation radish460-490
Emrawa                                          0.065    1998 35.5608  32.405Chalky Limestone                    Shallow Soil           350-400
Ethnaibeh                                       0.065    1998 35.5444 32.4128Chalky Limestone                    Eroded Soil            300-350
Eidon                                           4.62    2004 35.5113 32.3125Chalky Limestone                    Terra Rossa/Heavy Clay 600-720
Eidon                                           9.35    2004 35.5113 32.3225Chalky Limestone                    Terra Rossa/Heavy Clay 600-720
Attayba                                         0.051    2021 35.4259 32.3234Turonian Limestone                  Terra Rossa            350-500
Mazar shamali                                   0.02    2021 35.4742 32.2756Turonian Limestone                  Terra Rossa            750-850
Anjarah                                         3.94    2017 35.4527 32.1818Limestone/Marl                      Terra Rossa/Shallow    850-950
Ajloun city                                     3.63    2017 35.4602 32.2029Limestone/Marl                      Terra Rossa            750-850
Kufranjah                                       4.22    2017 35.4209 32.1743Limestone/Marl                      Alluvial Soils                400
Ayn jana                                        4.01    2017 35.4532 32.2024Limestone/Dolomite                  Terra Rossa            1000-1100
Ibbin                                           4.84    2017 35.4845 32.2135Dolomitic Limestone                 Terra Rossa            1050-1100
Ballas                                          3.21    2017  35.421 32.1545Limestone/Shale                     Terra Rossa                   700
Ayn Albustan                                    4.19    2017 35.4314 32.1815Limestone/Marl                      Alluvial Terra Rossa          650
Mafraq                                          0.05    2004 36.1208 32.2028Limestone                           Desert soil                   700
jerash                                          0.048    2004 35.5344  32.162Limestone                           Terra Rossa                   600
Ajlun                                           0.04    2004 35.4505 32.1954Limestone/Marl                      Terra Rossa            750-850
Madaba                                          0.092    2004 35.4735 31.4303Limestone/Marl                      Terra Rossa            750-820
Salt                                            0.046    2004 35.4335 32.0155Limestone/Marl                      Terra Rossa            700-1100
Tafila                                          0.047    2004 35.3656 30.4954Limestone/Phosphate/Basalt          Terra Rossa            400-1000
Karak                                           0.099    2004 35.4216 31.1054Limestone/Phosphate/Marl            Terra Rossa/Xerosols   800-1100
Ma'an                                           0.096    2004 35.4324 30.1058Dolomitic Limestone/Graniet/PhosphatYermosols/Aridisols    1000-1500
Aqaba                                           0.029    2004 35.0029 29.3152Graniet                             Granitic Sands               1000
Mutah-Karak                                     0.12    2015 35.4146 31.0532Dolomite Limestone                  Terra Rossa                  1100
Madden-Karak                                    0.1    2015 35.4357 31.0659Chalky Marl                         Terra Rossa                  1050
Rakeen-Karak                                    0.327    2015 35.4218 31.1329Limestone                           Terra Rossa                  1000
Barada-Karak                                    0.356    2015 35.4032 31.1243Graniet                             Dry sand                     1000
Adder-Karak                                     0.22    2015 35.4533  31.122Limestone/Phosphate                 Terra Rossa                   950
Al-Haweya-Karak                                 0.11    2015 35.4349 31.0101Limestone/Clay                      Terra Rossa                   920
Al-smakia-Karak                                 0.355    2015 35.4755 31.1816limestone/Graniet                   Terra Rossa                   980
Ma'an - Shoubak                                 4.037    1998 35.3325 30.3103Limestone/Phosphate                 Yellowish Brown Thin         1400
"""

# Data cleaning functions
def clean_elevation(elev_str):
    if pd.isna(elev_str):
        return np.nan
    elev_str = str(elev_str).strip()
    if '-' in elev_str and 'i' not in elev_str:
        try:
            parts = elev_str.split('-')
            if len(parts) == 2:
                return (float(parts[0]) + float(parts[1])) / 2
        except:
            pass
    cleaned = re.sub(r'[^\d.]', '', elev_str)
    try:
        return float(cleaned) if cleaned else np.nan
    except:
        return np.nan

def parse_raw_data(raw_text):
    lines = raw_text.strip().split('\n')
    data_rows = []
    for line in lines[1:]:
        if line.strip() == '':
            continue
        parts = re.split(r'\s{2,}', line.strip())
        if len(parts) >= 8:
            try:
                data_rows.append({
                    'Location': parts[0].strip(),
                    'Radon': float(re.search(r'[\d.]+', parts[1]).group()),
                    'Year': int(re.search(r'[\d.]+', parts[2]).group()),
                    'X': float(re.search(r'[\d.]+', parts[3]).group()),
                    'Y': float(re.search(r'[\d.]+', parts[4]).group()),
                    'Rock': parts[5].strip(),
                    'Soil': parts[6].strip(),
                    'Elevation': clean_elevation(parts[7])
                })
            except:
                pass
    return pd.DataFrame(data_rows)

# Parse and clean data
df = parse_raw_data(raw_data)
df = df.dropna(subset=['Radon', 'X', 'Y', 'Year'])
df = df[df['Radon'] <= 100]
df['Elevation'].fillna(df['Elevation'].median(), inplace=True)

# Add region classification
def assign_region(x, y):
    if 35.45 <= x <= 35.55 and 32.28 <= y <= 32.38:
        return 'Irbid'
    elif 35.4 <= x <= 35.6 and 32.1 <= y <= 32.5:
        return 'Northern'
    elif 36.0 <= x <= 36.3 and 32.1 <= y <= 32.3:
        return 'Mafraq'
    else:
        return 'Other'

df['Region'] = df.apply(lambda row: assign_region(row['X'], row['Y']), axis=1)

print(f"✓ Data cleaned: {len(df)} measurements")
print(f"✓ Mean radon: {df['Radon'].mean():.2f} ± {df['Radon'].std():.2f} KBq/m³")
print(f"✓ Radon range: {df['Radon'].min():.2f} - {df['Radon'].max():.2f} KBq/m³")
print(f"✓ Irbid measurements: {len(df[df['Region'] == 'Irbid'])}")
print(f"✓ Regions: {df['Region'].nunique()}")
print(f"\nData Preview:")
print(df.head(10))

## STEP 3: Machine Learning Model Development

In [ ]:
print("\n" + "="*80)
print(" "*20 + "STEP 2: MACHINE LEARNING MODEL DEVELOPMENT")
print("="*80)

# Prepare features
le_rock = LabelEncoder()
le_soil = LabelEncoder()
df['Rock_encoded'] = le_rock.fit_transform(df['Rock'].fillna('Unknown'))
df['Soil_encoded'] = le_soil.fit_transform(df['Soil'].fillna('Unknown'))
df['Distance_from_Irbid'] = np.sqrt((df['X'] - 35.505)**2 + (df['Y'] - 32.3309)**2)

features = ['X', 'Y', 'Elevation', 'Rock_encoded', 'Soil_encoded', 'Year', 'Distance_from_Irbid']
X = df[features].fillna(df[features].mean())
y = df['Radon'].astype(float)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeatures: {features}")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train models
models = {}
results = []

print("\nTraining models...\n")

# Random Forest
print("🔄 Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
models['Random Forest'] = rf
print(f"  ✓ R²: {r2_rf:.4f}, RMSE: {rmse_rf:.4f}, MAE: {mae_rf:.4f}")

# Gradient Boosting
print("🔄 Training Gradient Boosting...")
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
r2_gb = r2_score(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb = mean_absolute_error(y_test, y_pred_gb)
models['Gradient Boosting'] = gb
print(f"  ✓ R²: {r2_gb:.4f}, RMSE: {rmse_gb:.4f}, MAE: {mae_gb:.4f}")

# AdaBoost
print("🔄 Training AdaBoost...")
ab = AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
ab.fit(X_train, y_train)
y_pred_ab = ab.predict(X_test)
r2_ab = r2_score(y_test, y_pred_ab)
rmse_ab = np.sqrt(mean_squared_error(y_test, y_pred_ab))
mae_ab = mean_absolute_error(y_test, y_pred_ab)
models['AdaBoost'] = ab
print(f"  ✓ R²: {r2_ab:.4f}, RMSE: {rmse_ab:.4f}, MAE: {mae_ab:.4f}")

# Ridge Regression
print("🔄 Training Ridge Regression...")
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
models['Ridge'] = ridge
print(f"  ✓ R²: {r2_ridge:.4f}, RMSE: {rmse_ridge:.4f}, MAE: {mae_ridge:.4f}")

print("\n✓ All models trained!")

## STEP 4: Feature Importance & Model Analysis

In [ ]:
print("="*80)
print(" "*25 + "STEP 3: VISUALIZATIONS")
print("="*80)

# Feature Importance
plt.figure(figsize=(10, 6))
feature_names = ['Longitude', 'Latitude', 'Elevation', 'Rock Type', 'Soil Type', 'Year', 'Distance']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_names)))
plt.barh(range(len(indices)), importances[indices], color=colors)
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance for Radon Prediction\n(Random Forest Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Feature importance plot generated")

print("\nFeature Importance Ranking:")
for i, idx in enumerate(indices, 1):
    print(f"  {i}. {feature_names[idx]:20s} : {importances[idx]:.4f}")

## STEP 5: Residual Analysis

In [ ]:
# Residual Analysis
y_pred = rf.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6, s=80, edgecolors='k')
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Radon (KBq/m³)', fontsize=11)
axes[0, 0].set_ylabel('Residuals', fontsize=11)
axes[0, 0].set_title('Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Histogram of Residuals
axes[0, 1].hist(residuals, bins=15, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 1].set_xlabel('Residuals', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Q-Q Plot
from scipy.stats import probplot
probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normality Check)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Actual vs Predicted
axes[1, 1].scatter(y_test, y_pred, alpha=0.6, s=80, edgecolors='k')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Actual Radon (KBq/m³)', fontsize=11)
axes[1, 1].set_ylabel('Predicted Radon (KBq/m³)', fontsize=11)
axes[1, 1].set_title('Actual vs Predicted Values', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Residual analysis plot generated")

print(f"\nResidual Statistics:")
print(f"  Mean: {residuals.mean():.4f}")
print(f"  Std Dev: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## STEP 6: Geological Analysis

In [ ]:
print("="*80)
print(" "*25 + "STEP 4: GEOLOGICAL ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Rock type analysis
rock_stats = df.groupby('Rock')['Radon'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False).head(8)
axes[0, 0].barh(range(len(rock_stats)), rock_stats['mean'].values, 
                xerr=rock_stats['std'].values, capsize=5, alpha=0.7, color='coral')
axes[0, 0].set_yticks(range(len(rock_stats)))
axes[0, 0].set_yticklabels(rock_stats.index, fontsize=10)
axes[0, 0].set_xlabel('Average Radon (KBq/m³)', fontsize=11)
axes[0, 0].set_title('Rock Type Impact on Radon Levels\n(with 95% CI)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Soil type analysis
soil_stats = df.groupby('Soil')['Radon'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False).head(8)
axes[0, 1].barh(range(len(soil_stats)), soil_stats['mean'].values,
                xerr=soil_stats['std'].values, capsize=5, alpha=0.7, color='lightgreen')
axes[0, 1].set_yticks(range(len(soil_stats)))
axes[0, 1].set_yticklabels(soil_stats.index, fontsize=10)
axes[0, 1].set_xlabel('Average Radon (KBq/m³)', fontsize=11)
axes[0, 1].set_title('Soil Type Impact on Radon Levels\n(with 95% CI)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Regional analysis
region_stats = df.groupby('Region')['Radon'].agg(['mean', 'std']).sort_values('mean', ascending=False)
axes[1, 0].bar(region_stats.index, region_stats['mean'].values, 
              yerr=region_stats['std'].values, capsize=5, alpha=0.7, color='lightblue')
axes[1, 0].set_ylabel('Average Radon (KBq/m³)', fontsize=11)
axes[1, 0].set_title('Radon Concentration by Region\n(with Std Dev)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Elevation analysis
axes[1, 1].scatter(df['Elevation'], df['Radon'], alpha=0.6, s=80, edgecolors='k', c=df['Radon'], cmap='RdYlGn_r')
z = np.polyfit(df['Elevation'], df['Radon'], 1)
p = np.poly1d(z)
x_trend = np.linspace(df['Elevation'].min(), df['Elevation'].max(), 100)
axes[1, 1].plot(x_trend, p(x_trend), "r--", linewidth=2, label=f'Trend: y={z[0]:.4f}x+{z[1]:.2f}')
axes[1, 1].set_xlabel('Elevation (m)', fontsize=11)
axes[1, 1].set_ylabel('Radon (KBq/m³)', fontsize=11)
axes[1, 1].set_title('Elevation vs Radon Concentration', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('Radon (KBq/m³)', fontsize=10)

plt.tight_layout()
plt.savefig('geological_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Geological analysis plot generated")

## STEP 7: Final Summary Report

In [ ]:
print("\n" + "="*80)
print(" "*20 + "COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 DATASET OVERVIEW")
print("─" * 80)
print(f"  Total measurements: {len(df)}")
print(f"  Geographic regions: {df['Region'].nunique()}")
print(f"  Rock types: {df['Rock'].nunique()}")
print(f"  Soil types: {df['Soil'].nunique()}")
print(f"  Year range: {int(df['Year'].min())} - {int(df['Year'].max())}")
print(f"  Time span: {int(df['Year'].max() - df['Year'].min())} years")

print(f"\n🔬 RADON CONCENTRATION STATISTICS")
print("─" * 80)
print(f"  Mean: {df['Radon'].mean():.2f} KBq/m³")
print(f"  Median: {df['Radon'].median():.2f} KBq/m³")
print(f"  Std Dev: {df['Radon'].std():.2f} KBq/m³")
print(f"  Min: {df['Radon'].min():.2f} KBq/m³")
print(f"  Max: {df['Radon'].max():.2f} KBq/m³")
print(f"  Q1 (25%): {df['Radon'].quantile(0.25):.2f} KBq/m³")
print(f"  Q3 (75%): {df['Radon'].quantile(0.75):.2f} KBq/m³")

print(f"\n🌍 IRBID CITY ANALYSIS")
print("─" * 80)
irbid_data = df[df['Region'] == 'Irbid']
if len(irbid_data) > 0:
    print(f"  Measurements: {len(irbid_data)}")
    print(f"  Mean radon: {irbid_data['Radon'].mean():.2f} KBq/m³")
    print(f"  Range: {irbid_data['Radon'].min():.2f} - {irbid_data['Radon'].max():.2f} KBq/m³")
    print(f"  Status: {'ELEVATED' if irbid_data['Radon'].mean() > 2 else 'NORMAL'} (WHO limit: 0.1 KBq/m³)")

print(f"\n🤖 MACHINE LEARNING MODEL PERFORMANCE")
print("─" * 80)
print(f"  Model             R² Score   RMSE (KBq/m³)   MAE (KBq/m³)")
print(f"  ─" * 60)
print(f"  Random Forest     {r2_rf:.4f}      {rmse_rf:.4f}         {mae_rf:.4f}")
print(f"  Gradient Boost    {r2_gb:.4f}      {rmse_gb:.4f}         {mae_gb:.4f}")
print(f"  AdaBoost          {r2_ab:.4f}      {rmse_ab:.4f}         {mae_ab:.4f}")
print(f"  Ridge Regression  {r2_ridge:.4f}      {rmse_ridge:.4f}         {mae_ridge:.4f}")

print(f"\n🏆 BEST MODEL: Random Forest")
print("─" * 80)
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='r2')
print(f"  Cross-validation R² (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Test R²: {r2_rf:.4f}")
print(f"  Test RMSE: {rmse_rf:.4f} KBq/m³")
print(f"  Test MAE: {mae_rf:.4f} KBq/m³")
print(f"  Interpretation: Model explains {r2_rf*100:.1f}% of radon variation")
print(f"  Typical prediction error: ±{rmse_rf:.1f} KBq/m³")

print(f"\n🔍 TOP PREDICTIVE FEATURES")
print("─" * 80)
for i, idx in enumerate(indices[:5], 1):
    print(f"  {i}. {feature_names[idx]:20s} : {importances[idx]:.1%}")

print(f"\n🌡️ HIGH-RADON LOCATIONS (Top 5)")
print("─" * 80)
top_radon = df.nlargest(5, 'Radon')[['Location', 'Radon', 'Rock', 'Region']]
for i, (idx, row) in enumerate(top_radon.iterrows(), 1):
    print(f"  {i}. {row['Location']:30s} | {row['Radon']:7.2f} KBq/m³ | {row['Rock']:20s}")

print(f"\n⚠️ RADON HEALTH GUIDELINES (WHO)")
print("─" * 80)
print(f"  Safe level:       < 0.1 KBq/m³")
print(f"  Moderate risk:    0.1 - 0.4 KBq/m³")
print(f"  High risk:        0.4 - 2.0 KBq/m³")
print(f"  Very high risk:   > 2.0 KBq/m³")
print(f"\n  Jordan average:   {df['Radon'].mean():.2f} KBq/m³ (ELEVATED)")
print(f"  Irbid average:    {irbid_data['Radon'].mean():.2f} KBq/m³ (VERY ELEVATED)")

print("\n" + "="*80)
print(" "*25 + "✓ ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated visualizations:")
print("  ✓ feature_importance.png")
print("  ✓ residual_analysis.png")
print("  ✓ geological_analysis.png")
print("\nTo download: Right-click on image → Save image as")

## Download Results

Your analysis is complete! To download the generated charts:

1. **Feature Importance Chart** - Shows which geological factors matter most
2. **Residual Analysis** - Model diagnostic plots
3. **Geological Analysis** - Rock/soil/region/elevation impacts

All images are displayed above. Right-click any image and select "Save image as" to download.